### Read Data Sources

In [0]:
import pyspark.sql.functions as func

# Read the dataframe
df_sales = spark.read.table(
    'samples.bakehouse.sales_transactions'
)

df_sales_extended = spark.read.table(
    'workspace.sales_db_raw.sales_extended'
)

df = df_sales.unionByName(
    df_sales_extended
)
print(df_sales.count())
print(df.count())
display(df)

### Projections and filters

In [0]:
df = df.select(
    'dateTime',
    func.year('dateTime').alias('year'),
    func.month('dateTime').alias('month'),
    func.day('datetime').alias('day'),
    'transactionID',
    'customerID',
    'franchiseID',
    'product',
    'quantity',
    'totalPrice',
)

df_transaction_first_half = df.filter(
    func.col('day') <= 15
)

display(df_transaction_first_half)

Databricks visualization. Run in Databricks to view.

### Renaming, adding, and dropping columns

In [0]:
df_cus = spark.read.table(
    'samples.bakehouse.sales_customers'
).drop(
    'first_name',
    'last_name',
    'email_address',
    'phone_number',
    'address',
    'state',
    'continent'
)

df_cus = df_cus.withColumnRenamed(
    'postal_zip_code',
    'customerZipcode'
).withColumnRenamed(
    'city',
    'customerCity'
).withColumnRenamed(
    'country',
    'customerCountry'
)

df_cus = df_cus.withColumn(
    'customerGender',
    func.when(
        func.col('gender') == 'female',
        'F'
    ).otherwise(
        'M'
    )
).drop('gender')


display(df_cus)

### Joins

In [0]:
# Read franchises
df_fr = spark.read.table(
    'samples.bakehouse.sales_franchises'
)

# Select columns of interest

df_fr = df_fr.select(
    'franchiseID',
    func.col('name').alias('franchiseName'),
    func.col('city').alias('franchiseCity'),
    func.col('country').alias('franchiseCountry'),
    func.col('zipcode').alias('franchiseZipcode'),
    func.col('size').alias('franchiseSize'),
    'supplierID'
)

# Read supliers
df_sup = spark.read.table(
    'samples.bakehouse.sales_suppliers'
)

# Select columns of interest
df_sup = df_sup.select(
    'supplierID',
    func.col('name').alias('supplierName'),
    'ingredient',
    func.col('size').alias('supplierSize')
)

# Join Franchise with Supplier
df_fr_sup = df_fr.join(
    df_sup,
    on='supplierID',
    how='left'
)

display(df_fr_sup)


**Exercise**: Join transactions with customers, franchise and suppliers and call it **df_r**

In [0]:
df_r = df\
    .join(df_cus, on='customerID', how='left') \
    .join(df_fr_sup, on='franchiseID', how='left')

display(df_r)

# **Write Preprocessed Table**

In [0]:
df_write = df_r.select(
    'dateTime',
    'year',
    'month',
    'day',
    'transactionID',
    'customerID',
    'franchiseID',
    'supplierID',
    'product',
    'ingredient',
    'quantity',
    'totalPrice',
    'customerCity',
    'customerCountry',
    func.col('customerZipcode').try_cast('string'),
    'customerGender',
    'franchiseName',
    'franchiseCity',
    'franchiseCountry',
    func.col('franchiseZipcode').try_cast('string'),
    'franchiseSize',
    'supplierName',
    'supplierSize'
).distinct()

In [0]:
# Create a database and a table
spark.sql(
    '''
        CREATE DATABASE IF NOT EXISTS sales_db_gold
    '''
)

spark.sql(
    '''
        CREATE TABLE IF NOT EXISTS sales_db_gold.sales
        (
            dateTime TIMESTAMP,
            year INT,
            month INT,
            day INT,
            transactionID BIGINT,
            customerID BIGINT,
            franchiseID BIGINT,
            supplierID BIGINT,
            product STRING,
            ingredient STRING,
            quantity BIGINT,
            totalPrice BIGINT,
            customerCity STRING,
            customerCountry STRING,
            customerZipcode STRING,
            customerGender STRING,
            franchiseName STRING,
            franchiseCity STRING,
            franchiseCountry STRING,
            franchiseZipcode STRING,
            franchiseSize STRING,
            supplierName STRING,
            supplierSize STRING)
    '''
)



In [0]:
df_write.write.mode('overwrite').saveAsTable('sales_db_gold.sales')